# ***Prueba Control continuo***


Nombre equipo: *Shampoo*

In [ ]:
%matplotlib inline


import ipywidgets
import numpy
import sympy

sympy.init_printing()

import matplotlib.pyplot as plt
try:
  from control.matlab import *
  import control
except:
  !pip install control
  from control.matlab import *
  import control

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 549.6/549.6 kB 8.6 MB/s eta 0:00:00


In [ ]:
# Importamos SymPy para el manejo simbólico
import sympy as sp

# Definimos las constantes y parámetros según la imagen
J = 0.01        # kg*m^2
b = 0.1         # N*m*s
Ke = 0.01       # V / (rad/s)
Kt = 0.01       # N*m / A
R = 0.9         # Ohm
L = 0.5         # H

# Definimos las variables del sistema
theta, theta_dot, theta_ddot = sp.symbols('theta theta_dot theta_ddot')  # Angular variables
i, i_dot = sp.symbols('i i_dot')  # Current and its derivative
V = 12  # Voltage fixed at 12 volts

# Ecuaciones diferenciales del sistema a partir de la imagen
# Ecuación 1: J * theta_ddot + b * theta_dot = Ke * i
eq1 = sp.Eq(J * theta_ddot + b * theta_dot, Ke * i)

# Ecuación 2: L * i_dot + R * i = V - Kt * theta_dot
eq2 = sp.Eq(L * i_dot + R * i, V - Kt * theta_dot)

# Calculamos las derivadas parciales para la linealización
# Punto de equilibrio: theta_dot = 0, i = 0
equilibrium = {
    theta: 0,
    theta_dot: 0,
    theta_ddot: 0,
    i: 0,
    i_dot: 0
}

# Linealizamos cada ecuación alrededor del punto de equilibrio

# Derivadas parciales de la primera ecuación
dF1_dtheta_dot = sp.diff(eq1.lhs, theta_dot).subs(equilibrium)
dF1_di = sp.diff(eq1.lhs, i).subs(equilibrium)

# Derivadas parciales de la segunda ecuación
dF2_dtheta_dot = sp.diff(eq2.lhs, theta_dot).subs(equilibrium)
dF2_di = sp.diff(eq2.lhs, i).subs(equilibrium)

# Ecuación linealizada combinada para theta_ddot
# Usando las derivadas parciales calculadas
theta_d, theta_dot_d, i_d = sp.symbols('theta_d theta_dot_d i_d')

# Expresiones para las ecuaciones linealizadas
linearized_eq1 = J * theta_ddot + b * theta_dot_d - Ke * i_d  # De la primera ecuación
linearized_eq2 = L * i_dot + R * i_d - (V - Kt * theta_dot_d)  # De la segunda ecuación

# Sustituimos V = 12 y combinamos las ecuaciones
linearized_combined = sp.Eq(linearized_eq1 - linearized_eq2, 0).simplify()

# Mostramos la ecuación linealizada combinada
print("Ecuación linealizada combinada:")
sp.pprint(linearized_combined)

Ecuación linealizada combinada:
0.91⋅i_d + 0.5⋅i_dot - 0.01⋅θ_ḋ - 0.09⋅θ_dot_d = 12.0


In [ ]:
%matplotlib inline


import ipywidgets
import numpy
import sympy

sympy.init_printing()

import matplotlib.pyplot as plt
try:
  from control.matlab import *
  import control
except:
  !pip install control
  from control.matlab import *
  import control

# Importamos la librería Sympy
from sympy import *

# Definimos las constantes del sistema y las variables
M, m, l, g, Fw = symbols("M m l g Fw")          # Constantes del sistema
theta, thetap, thetapp, u, xpp = symbols(("\\theta,\dot{\\theta},\ddot{\\theta},u,\ddot{x}"))  # Variables del sistema

# Definimos las ecuaciones del sistema
# Ecuaciones no lineales

eq1 = Eq((M + m) * xpp - l * m * sin(theta) * thetap**2 + l * m * cos(theta) * thetapp, u + Fw)
eq1

eq2 = Eq(m * xpp * cos(theta) + l * m * thetapp, m * g * sin(theta) + Fw * cos(theta))
eq2

# Despejamos x_ddot de la primera ecuación
x_ddot_sol = solve(eq1, xpp)[0]

# Sustituimos x_ddot en la segunda ecuación para obtener la ecuación solo en theta_ddot
eq2_sub = eq2.subs(xpp, x_ddot_sol).simplify()

# Punto de linealización: Estado estacionario
# Definimos las condiciones de linealización
theta_ss, theta_dot_ss = symbols("theta_{ss} theta_dot_{ss}")
condiciones = {
    theta: 0,          # Theta en el punto de equilibrio
    thetap: 0,      # Velocidad angular inicial
    thetapp: 0,     # Aceleración angular inicial
    Fw: 0,             # Sin perturbación del viento
    u: 0               # Control nulo en equilibrio
}

# Sustitución de condiciones para obtener una expresión en theta_ddot
eq2_simplified = eq2_sub.subs(condiciones).simplify()

# Despejamos theta_ddot en función de las variables relevantes
try:
    theta_ddot_sol = solve(eq2_sub, thetapp)[0]  # Usamos la ecuación sin condiciones
    eq3 = Eq(thetapp, theta_ddot_sol)
except IndexError:
    print("No se encontró una solución directa para theta_ddot.")
eq3

# Expansión en serie de Taylor
# Definimos la función para aplicar la expansión en serie de Taylor
F = eq3.rhs
F

# La función evaluada en el punto de linealización
Fss = F.subs(condiciones)
Fss

# Derivada de la función con respecto a theta evaluada en el punto de linealización
dF_theta = diff(F, theta)
dF_theta_ss = dF_theta.subs(condiciones)
dF_theta_ss

# Derivada de la función con respecto a theta_dot evaluada en el punto de linealización
dF_theta_dot = diff(F, thetap)
dF_theta_dot_ss = dF_theta_dot.subs(condiciones)
dF_theta_dot_ss

# Derivada de la función con respecto a u evaluada en el punto de linealización
dF_u = diff(F, u)
dF_u_ss = dF_u.subs(condiciones)
dF_u_ss

# Generando la expresión completa de Taylor
F_taylor_expansion = Fss + dF_theta_ss * (theta - theta_ss) + dF_theta_dot_ss * (thetap) + dF_u_ss * (u)
F_taylor_expansion

# Usando variables desviadas
theta_d, u_d = symbols("theta_d u_d")
eq4 = Eq(thetapp, dF_theta_ss * theta_d + dF_theta_dot_ss * thetap + dF_u_ss * u_d)
eq4

# Remplazando los valores numéricos
parametros = {
    M: 2.4,
    m: 0.23,
    l: 0.36,
    g: 9.81
}

# Ecuación linealizada con los valores específicos del sistema
eq5 = eq4.subs(parametros).evalf()
eq5

\ddot{\theta} = 29.8614583333333⋅θ_d - 1.15740740740741⋅u_d